In [1]:
tabla='ctpco10'
col_essi='fec_pro'
col_tabla='proconfec'
essi='essi_dat_cex004_etl'

In [2]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
import oracledb
import sys
from sqlalchemy import text

In [3]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [4]:
from datetime import datetime
from sqlalchemy import text


fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=6", con=connection2)
fecha= fecha.iloc[0, 0]
print(fecha)

now = datetime.now()

query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=6"

c1= text(query)
connection2.execute(c1)

2023-05-24


/tmp/ipykernel_107767/2225831687.py:14: RemovedIn20Warning: Deprecated API features detected! These feature(s) are not compatible with SQLAlchemy 2.0. To prevent incompatible upgrades prior to updating applications, ensure requirements files are pinned to "sqlalchemy<2.0". Set environment variable SQLALCHEMY_WARN_20=1 to show all deprecation warnings.  Set environment variable SQLALCHEMY_SILENCE_UBER_WARNING=1 to silence this message. (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  connection2.execute(c1)


In [7]:
cas = pd.read_sql_query(f"SELECT id_red,cod_cas,des_cas FROM dim_cas where id_red is not null", con=connection2)
red = pd.read_sql_query(f"SELECT id_red,cod_red,des_red FROM dim_red", con=connection2)
cas_red=pd.merge(cas,red,how='left',on='id_red')
cas_red.to_sql(name='tmp_cas_red', con=engine1, if_exists='replace', index=False)

servicios = pd.read_sql_query(f"SELECT cod_ser,des_ser FROM dim_servicios", con=connection2)
servicios.to_sql(name='tmp_servicios', con=engine1, if_exists='replace', index=False)

areas = pd.read_sql_query(f"SELECT cod_are,des_are FROM dim_areas", con=connection2)
areas.to_sql(name='tmp_areas', con=engine1, if_exists='replace', index=False)

subacti = pd.read_sql_query(f"SELECT cod_sub,des_sub FROM dim_subacti", con=connection2)
subacti.to_sql(name='tmp_subacti', con=engine1, if_exists='replace', index=False)


activi = pd.read_sql_query(f"SELECT cod_act,des_act FROM dim_activi", con=connection2)
activi.to_sql(name='tmp_activi', con=engine1, if_exists='replace', index=False)

36

query_count_before = f"SELECT COUNT(*) FROM {essi}"
cant_antes = connection1.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {essi} antes de la inserción: {cant_antes}")

In [8]:
borrando=f"DELETE FROM {essi} WHERE {col_essi} >='{fecha}'"
borrado = connection1.execute(borrando)

In [11]:
query1=f"""

INSERT INTO {essi}
(cod_con,cod_act,cod_sub,cod_are,cit_ate,cit_eli,cit_rep,cup_dia,cup_ref,cup_vol,cup_int,cup_rec,top_adi,top_nue,top_ref,top_ter,dem_ins,oto_ref,oto_vol,oto_int,oto_rec,oto_top_adi,oto_top_nue,oto_top_ref,oto_top_ter,pac_hor,cod_cas,est_com,reg_cod,fec_pro,ori_cas,num_doc,cod_ser,tip_doc,hor_fin,hor_ini) 

SELECT 
concod,proconactcod,proconactespcod,proconarehoscod,proconcancitate,proconcanciteli,proconcancitrep,proconcancupcitdia,proconcancupcitref,proconcancupcitvol,proconcancupinte,proconcancupreci,proconcancuptopadic,proconcancuptopnuev,proconcancuptoprefe,proconcancuptopterc,proconcandemins,proconcanotorcitref,proconcanotorcitvol,proconcanotorinte,proconcanotorreci,proconcanotortopadic,proconcanotortopnuev,proconcanotortoprefe,proconcanotortopterc,proconcanpachor,proconcenasicod,proconestcom,proconestregcod,proconfec,proconoricenasicod,proconperasisdocidennum,proconservhoscod,procontipdocidenpercod,proconturhorfin,proconturhorini




FROM dblink('dbname=dl_essi user=ugaddba001ir password=U64dING23', 'SELECT * FROM {tabla} WHERE {tabla}.{col_tabla} >=''{fecha}''')

AS tmp_tbl(
	proconoricenasicod  character(1),
	proconcenasicod  character(3),
	proconarehoscod  character(2),
	proconservhoscod  character(3),
	proconactcod  character(2),
	proconactespcod  character(3),
	procontipdocidenpercod  character(1),
	proconperasisdocidennum  character(10),
	proconfec  date,
	proconturhorini  timestamp,
	proconturhorfin  timestamp,
	concod  character(5),
	proconcancupcitvol  numeric(3,0),
	proconcancupreci  numeric(3,0),
	proconcancupinte  numeric(3,0),
	proconcancupcitdia  numeric(3,0),
	proconcancuptopnuev  numeric(3,0),
	proconcancuptopadic  numeric(3,0),
	proconcancuptoprefe  numeric(3,0),
	proconcancuptopterc  numeric(3,0),
	proconcanotorcitvol  numeric(3,0),
	proconcanotorreci  numeric(3,0),
	proconcanotorinte  numeric(3,0),
	proconcanotortopnuev  numeric(3,0),
	proconcanotortopadic  numeric(3,0),
	proconcanotortoprefe  numeric(3,0),
	proconcanotortopterc  numeric(3,0),
	proconcanciteli  numeric(3,0),
	proconcancitate  numeric(3,0),
	proconcancitrep  numeric(3,0),
	proconcandemins  numeric(3,0),
	proconestcom  character(1),
	proconestregcod  character(1),
	proconcancupcitref  numeric(3,0),
	proconcanotorcitref  numeric(3,0),
	proconcanpachor  numeric(2,0)
);
"""
c2= text(query1)
cursor=connection1.execute(c2)




query_count_after = f"SELECT COUNT(*) FROM {essi}"
cant_despues = connection1.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla {essi} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

### Trayendo maestros

In [12]:
query1=f"""
ALTER TABLE tmp_cas_red 
ALTER COLUMN id_red TYPE CHARACTER (2),
ALTER COLUMN cod_cas TYPE CHARACTER (3),
ALTER COLUMN cod_red TYPE CHARACTER (2),
ALTER COLUMN des_red TYPE CHARACTER (60),
ALTER COLUMN des_cas TYPE CHARACTER (50);

UPDATE {essi} 
SET 
des_cas= t.des_cas,
des_red= t.des_red,
cod_red= t.cod_red
FROM tmp_cas_red t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_cas = t.cod_cas and {essi}.cod_cas IS NOT NULL and t.cod_cas IS NOT NULL ;



ALTER TABLE tmp_servicios
ALTER COLUMN cod_ser TYPE CHARACTER (3),
ALTER COLUMN des_ser TYPE CHARACTER (50);

UPDATE {essi} 
SET 
des_ser= t.des_ser

FROM tmp_servicios t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_ser = t.cod_ser and {essi}.cod_ser IS NOT NULL and t.cod_ser IS NOT NULL ;



ALTER TABLE tmp_areas
ALTER COLUMN cod_are TYPE CHARACTER (2),
ALTER COLUMN des_are TYPE CHARACTER (30);

UPDATE {essi} 
SET 
des_are= t.des_are

FROM tmp_areas t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_are = t.cod_are and {essi}.cod_are IS NOT NULL and t.cod_are IS NOT NULL ;




ALTER TABLE tmp_activi 
ALTER COLUMN cod_act TYPE CHARACTER (2),
ALTER COLUMN des_act TYPE CHARACTER (50);

UPDATE {essi} 
SET 
des_act= t.des_act

FROM tmp_activi t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_act = t.cod_act and {essi}.cod_act IS NOT NULL and t.cod_act IS NOT NULL ;


ALTER TABLE tmp_subacti
ALTER COLUMN cod_sub TYPE CHARACTER (3),
ALTER COLUMN des_sub TYPE CHARACTER (60);

UPDATE {essi} 
SET 
des_sub= t.des_sub

FROM tmp_subacti t 
WHERE {essi}.{col_essi} >='{fecha}' and {essi}.cod_sub = t.cod_sub;


DROP TABLE tmp_areas;
DROP TABLE tmp_servicios;
DROP TABLE tmp_cas_red;
DROP TABLE tmp_activi;
DROP TABLE tmp_subacti;


"""
c2= text(query1)
cursor=connection1.execute(c2)

In [ ]:
connection1.close()
connection2.close()
connection3.close()